In [1]:
!pip install imbalanced-learn

import os
import numpy as np
import pandas as pd
import joblib
from google.colab import files
from IPython.display import display, Markdown
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

RAW_PATH = '/cardekho.csv' if os.path.exists('/cardekho.csv') else '/content/cardekho.csv'

class FrequencyEncoder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.maps_ = [X.iloc[:, i].value_counts(normalize=True).to_dict() for i in range(X.shape[1])]
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        cols = [X.iloc[:, i].map(self.maps_[i]).fillna(0.0).astype(float).values for i in range(X.shape[1])]
        return np.column_stack(cols)

    def get_feature_names_out(self, input_features=None):
        return np.array([str(c) + '_freq' for c in input_features])

In [2]:
'''Q4. Write your work as four functions in the notebook: load_data(), clean(df), add_features(df), and build_pipeline(). Put each in its own cell. The rest of the notebook only calls them.'''
def load_data():
    df = pd.read_csv(RAW_PATH)
    df.columns = df.columns.str.replace(r'\(.*?\)', '', regex=True).str.strip().str.lower()
    df = df.rename(columns={'fuel_type': 'fuel', 'transmission_type': 'transmission', 'car_name': 'name'})
    return df

In [3]:
def clean(df):
    d = df.copy()
    d = d.drop(columns=[c for c in ['torque', 'unnamed: 0'] if c in d.columns])
    for c in ['mileage', 'engine', 'max_power']:
        d[c] = pd.to_numeric(d[c].astype(str).str.extract(r'(\d+\.?\d*)')[0], errors='coerce')
    for c in ['fuel', 'seller_type', 'transmission', 'owner']:
        d[c] = d[c].astype(object).str.strip().str.replace(r'\s+', ' ', regex=True).str.title().replace({'Cng': 'CNG', 'Lpg': 'LPG'})
    d['name'] = d['name'].astype(object).str.strip().str.replace(r'\s+', ' ', regex=True)
    d.loc[d['km_driven'] <= 0, 'km_driven'] = np.nan
    d.loc[(d['year'] < 1980) | (d['year'] > 2026), 'year'] = np.nan
    for c in ['mileage', 'engine', 'max_power']:
        d.loc[d[c] <= 0, c] = np.nan
    d.loc[(d['seats'] < 2) | (d['seats'] > 14), 'seats'] = np.nan
    d = d[d['selling_price'] > 0]
    d = d.drop_duplicates().reset_index(drop=True)
    d['high_price'] = (d['selling_price'] >= 1_000_000).astype(int)
    return d

In [4]:
def add_features(df):
    d = df.copy()
    d['car_age'] = (2026 - d['year']).clip(lower=1)
    d['km_per_year'] = d['km_driven'] / d['car_age']
    d['power_per_cc'] = d['max_power'] / d['engine']
    d['log_km_driven'] = np.log1p(d['km_driven'])
    d['brand'] = d['name'].str.split().str[0]
    d['model'] = d['name'].str.split().str[:2].str.join(' ')
    premium = ['bmw', 'audi', 'mercedes-benz', 'jaguar', 'land', 'volvo', 'lexus', 'jeep', 'porsche', 'mini']
    d['is_premium'] = d['brand'].str.lower().isin(premium).astype(int)
    d = d.drop(columns=['name', 'year'])
    return d

In [5]:
'''Q4 (continued). build_pipeline()'''
def build_pipeline():
    num_cols = ['car_age', 'km_driven', 'log_km_driven', 'km_per_year', 'power_per_cc', 'mileage', 'engine', 'max_power', 'seats', 'is_premium']
    ohe_cols = ['fuel', 'seller_type', 'transmission']
    ord_cols = ['owner']
    freq_cols = ['brand', 'model']
    owner_order = ['Test Drive Car', 'First Owner', 'Second Owner', 'Third Owner', 'Fourth & Above Owner']
    preprocess = ColumnTransformer([
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
        ('ohe', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), ohe_cols),
        ('ord', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OrdinalEncoder(categories=[owner_order], handle_unknown='use_encoded_value', unknown_value=-1))]), ord_cols),
        ('freq', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', FrequencyEncoder())]), freq_cols)
    ], remainder='drop')
    return Pipeline([('preprocess', preprocess)])

In [6]:
'''Q1. Write a data dictionary as a table: every column, its meaning, its type, its unit, and its expected range.'''
raw = load_data()

dictionary = pd.DataFrame([
    ['name', 'Car brand, model and variant', 'text', 'none', 'free text'],
    ['year', 'Manufacturing year', 'integer', 'calendar year', '1980 to 2026'],
    ['selling_price', 'Listed selling price', 'integer', 'INR', '10,000 to 20,000,000'],
    ['km_driven', 'Odometer reading', 'integer', 'km', '1 to 500,000'],
    ['fuel', 'Fuel type', 'category', 'none', 'Diesel, Petrol, CNG, LPG'],
    ['seller_type', 'Type of seller', 'category', 'none', 'Individual, Dealer, Trustmark Dealer'],
    ['transmission', 'Gearbox type', 'category', 'none', 'Manual, Automatic'],
    ['owner', 'Ownership history', 'ordered category', 'none', 'Test Drive Car to Fourth & Above Owner'],
    ['mileage', 'Fuel efficiency', 'text with unit', 'kmpl (km/kg for CNG and LPG)', '5 to 50'],
    ['engine', 'Engine displacement', 'text with unit', 'CC', '600 to 4,000'],
    ['max_power', 'Maximum engine power', 'text with unit', 'bhp', '30 to 400'],
    ['torque', 'Engine torque with rpm', 'text with unit', 'Nm or kgm', '50 to 800 Nm'],
    ['seats', 'Number of seats', 'float', 'seats', '2 to 14']
], columns=['column', 'meaning', 'type', 'unit', 'expected range'])

display(dictionary[dictionary['column'].isin(raw.columns)])
print('Columns in the file but not in the dictionary:', sorted(set(raw.columns) - set(dictionary['column'])))
print(raw.shape)
print(raw.dtypes)

,column,meaning,type,unit,expected range
0,name,"Car brand, model and variant",text,none,free text
1,year,Manufacturing year,integer,calendar year,1980 to 2026
2,selling_price,Listed selling price,integer,INR,"10,000 to 20,000,000"
3,km_driven,Odometer reading,integer,km,"1 to 500,000"
4,fuel,Fuel type,category,none,"Diesel, Petrol, CNG, LPG"
5,seller_type,Type of seller,category,none,"Individual, Dealer, Trustmark Dealer"
6,transmission,Gearbox type,category,none,"Manual, Automatic"
7,owner,Ownership history,ordered category,none,Test Drive Car to Fourth & Above Owner
8,mileage,Fuel efficiency,text with unit,kmpl (km/kg for CNG and LPG),5 to 50
9,engine,Engine displacement,text with unit,CC,"600 to 4,000"


Columns in the file but not in the dictionary: []
(8128, 12)
name              object
year               int64
selling_price      int64
km_driven          int64
fuel              object
seller_type       object
transmission      object
owner             object
mileage          float64
engine           float64
max_power         object
seats            float64
dtype: object


In [ ]:
'''Q2. Produce a raw quality report covering missing values, duplicates, impossible values such as a negative age, inconsistent units, and inconsistent category labels. Give a count for each problem found.'''
parsed = {}
for c in ['mileage', 'engine', 'max_power']:
    parsed[c] = pd.to_numeric(raw[c].astype(str).str.extract(r'(\d+\.?\d*)')[0], errors='coerce')

label_issues = 0
for c in ['fuel', 'seller_type', 'transmission', 'owner']:
    s = raw[c].dropna().astype(str)
    label_issues += int((s != s.str.strip().str.replace(r'\s+', ' ', regex=True).str.title().replace({'Cng': 'CNG', 'Lpg': 'LPG'})).sum())

quality = {
    'missing cells (all columns)': int(raw.isna().sum().sum()),
    'duplicate rows': int(raw.duplicated().sum()),
    'km_driven <= 0': int((raw['km_driven'] <= 0).sum()),
    'year outside 1980-2026': int(((raw['year'] < 1980) | (raw['year'] > 2026)).sum()),
    'selling_price <= 0': int((raw['selling_price'] <= 0).sum()),
    'mileage/engine/max_power <= 0': int(sum((parsed[c] <= 0).sum() for c in parsed)),
    'seats outside 2-14': int(((raw['seats'] < 2) | (raw['seats'] > 14)).sum()),
    'mileage in km/kg (mixed unit)': int(raw['mileage'].astype(str).str.contains('km/kg', case=False).sum()),
    'torque in kgm (mixed unit)': int(raw['torque'].astype(str).str.contains('kgm', case=False).sum()),
    'category labels with spacing/case issues': label_issues
}

print('Missing values per column:')
print(raw.isna().sum()[raw.isna().sum() > 0])
print()
print(pd.Series(quality, name='count'))

In [7]:
'''Q3. Design at least 4 new features from the existing columns. For each one, write the single line of reasoning that justifies it.'''
reasoning = {
    'car_age': 'Age in years is more directly tied to depreciation than the raw year number.',
    'km_per_year': 'Total kilometres mean little without age; yearly usage separates heavy-use from lightly-used cars.',
    'power_per_cc': 'Power per unit of engine size shows how tuned or efficient the engine is, beyond raw power or size alone.',
    'log_km_driven': 'km_driven is heavily right-skewed, and the log gives a more symmetric version for later models.',
    'brand': 'The brand is the first word of the name, giving a usable category from free text.',
    'model': 'Brand plus model name captures the specific car line, a strong price signal that name alone hides in variants.',
    'is_premium': 'Premium brands sit in a different price band, so one flag gives a clean split.'
}
for k, v in reasoning.items():
    print(k, '->', v)

preview = add_features(clean(raw))
display(preview[list(reasoning.keys())].head())

car_age -> Age in years is more directly tied to depreciation than the raw year number.
km_per_year -> Total kilometres mean little without age; yearly usage separates heavy-use from lightly-used cars.
power_per_cc -> Power per unit of engine size shows how tuned or efficient the engine is, beyond raw power or size alone.
log_km_driven -> km_driven is heavily right-skewed, and the log gives a more symmetric version for later models.
brand -> The brand is the first word of the name, giving a usable category from free text.
model -> Brand plus model name captures the specific car line, a strong price signal that name alone hides in variants.
is_premium -> Premium brands sit in a different price band, so one flag gives a clean split.


,car_age,km_per_year,power_per_cc,log_km_driven,brand,model,is_premium
0,12.0,12125.000000,0.059295,11.887938,Maruti,Maruti Swift,0
1,12.0,10000.000000,0.069105,11.695255,Skoda,Skoda Rapid,0
2,20.0,7000.000000,0.052104,11.849405,Honda,Honda City,0
3,16.0,7937.500000,0.064470,11.751950,Hyundai,Hyundai i20,0
4,19.0,6315.789474,0.067951,11.695255,Maruti,Maruti Swift,0


In [8]:
'''Q5. Run the full path from the raw file to X_train, X_test, y_train, y_test in one final cell that calls those four functions. Print the shape of each.'''
df = load_data()
df = clean(df)
df = add_features(df)

y = df['high_price']
X = df.drop(columns=['high_price', 'selling_price'])

X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

pipeline = build_pipeline()
X_train = pd.DataFrame(pipeline.fit_transform(X_train_raw), columns=pipeline.get_feature_names_out(), index=X_train_raw.index)
X_test = pd.DataFrame(pipeline.transform(X_test_raw), columns=pipeline.get_feature_names_out(), index=X_test_raw.index)

print('X_train:', X_train.shape)
print('X_test :', X_test.shape)
print('y_train:', y_train.shape)
print('y_test :', y_test.shape)

X_train: (5540, 22)
X_test : (1386, 22)
y_train: (5540,)
y_test : (1386,)


In [9]:
'''Q6. Prove the output is clean using assert statements: no NaN remains, train and test have the same number of columns, and no column is constant.'''
assert not X_train.isna().any().any()
assert not X_test.isna().any().any()
assert X_train.shape[1] == X_test.shape[1]
assert (X_train.nunique() > 1).all()
print('All asserts passed')

All asserts passed


In [10]:
'''Q7. Record a before and after table: row count, column count, missing value count, and duplicate count, at the raw stage and at the final stage.'''
df_clean = clean(raw)
full = pd.concat([X_train, X_test])

before_after = pd.DataFrame({
    'stage': ['Raw file', 'After clean()', 'Final matrix'],
    'rows': [len(raw), len(df_clean), len(full)],
    'columns': [raw.shape[1], df_clean.shape[1], full.shape[1]],
    'missing values': [int(raw.isna().sum().sum()), int(df_clean.isna().sum().sum()), int(full.isna().sum().sum())],
    'duplicate rows': [int(raw.duplicated().sum()), int(df_clean.duplicated().sum()), int(full.duplicated().sum())]
})
display(before_after)

,stage,rows,columns,missing values,duplicate rows
0,Raw file,8128,12,878,1202
1,After clean(),6926,13,848,0
2,Final matrix,6926,22,0,493


In [11]:
'''Q8. Save the cleaned data as a CSV and the fitted pipeline with joblib. Download both from Colab.'''
cleaned = pd.concat([X_train, X_test])
cleaned['high_price'] = pd.concat([y_train, y_test])
cleaned['split'] = ['train'] * len(X_train) + ['test'] * len(X_test)

cleaned.to_csv('cleaned_data.csv', index=False)
joblib.dump(pipeline, 'pipeline.joblib')

files.download('cleaned_data.csv')
files.download('pipeline.joblib')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
'''NA'''

'NA'

In [14]:
'''Q10. Write one short paragraph on what is still wrong with this dataset that you could not fix.'''
display(Markdown('''
**What is still wrong**

Mileage is in kmpl for petrol and diesel cars but in km/kg for CNG and LPG cars, and I could not convert one to the other, so mileage is not fully comparable between fuel types. I also had to drop torque because it had mixed units and messy text. The prices are listing prices, not the final price the car sold for, and they are not adjusted for inflation. There is no car ID, so I can't tell if two identical rows are the same listing or two different cars, which means some rows I removed as duplicates may have been real. The model name is taken from the first two words of the name, so it is wrong for some cars. Also, engine, power, mileage and seats are often missing on the same rows, so filling them with the median is only a rough guess.
'''))


**What is still wrong**

Mileage is in kmpl for petrol and diesel cars but in km/kg for CNG and LPG cars, and I could not convert one to the other, so mileage is not fully comparable between fuel types. I also had to drop torque because it had mixed units and messy text. The prices are listing prices, not the final price the car sold for, and they are not adjusted for inflation. There is no car ID, so I can't tell if two identical rows are the same listing or two different cars, which means some rows I removed as duplicates may have been real. The model name is taken from the first two words of the name, so it is wrong for some cars. Also, engine, power, mileage and seats are often missing on the same rows, so filling them with the median is only a rough guess.


In [15]:
'''SUBMIT. README.md with the dataset link and the before and after table from question 7.'''
table = '| Stage | Rows | Columns | Missing values | Duplicate rows |\n|---|---|---|---|---|\n'
for _, r in before_after.iterrows():
    table += f"| {r['stage']} | {r['rows']} | {r['columns']} | {r['missing values']} | {r['duplicate rows']} |\n"

readme = f'''# Assignment 4: End-to-end data preparation

Dataset: CarDekho used cars, https://www.kaggle.com/datasets/nehalbirla/vehicle-dataset-from-cardekho

No model is trained. The notebook stops at a clean, model-ready matrix.

## Before and after

{table}
## Files

- assignment4_endtoend.ipynb
- cleaned_data.csv
- pipeline.joblib
'''
with open('README.md', 'w') as f:
    f.write(readme)

files.download('README.md')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>